# Gap 12 Analysis: B10/N10 NCC > ProtoNet Anomaly

On the B10/N10 split at 5-shot, NCC (Euclidean) achieves **97.40%** while ProtoNet (Euclidean) reaches only **95.77%** — a reversal from the B15/N5 split where ProtoNet leads.

This notebook investigates:
1. Per-episode paired difference distribution (NCC − ProtoNet)
2. Statistical significance of the reversal (Welch's t-test)
3. Whether specific novel classes drive the reversal
4. Comparison across all shot counts and distance metrics

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = 'white'

# Adjust root to point to project root (one level up from notebooks/)
ROOT = os.path.normpath(os.path.join(os.getcwd(), '..'))
RESULTS = os.path.join(ROOT, 'results')
print(f'Project root : {ROOT}')
print(f'Results dir  : {RESULTS}')

In [ ]:
def load_episodes(model, split, k_shot, distance='euclidean'):
    """Load per-episode accuracy CSV for a model/split/shot combination."""
    if model == 'ncc':
        path = os.path.join(
            RESULTS, 'baseline', 'resnet50', split, 'novel_ncc',
            f'{k_shot}shot_{distance}', 'episode_accuracies.csv'
        )
    elif model == 'protonet':
        results_split = 'base10_novel10' if '10' in split else 'base15_novel5'
        path = os.path.join(
            RESULTS, 'proto_net', results_split,
            f'{k_shot}shot_{distance}', 'episode_accuracies.csv'
        )
    else:
        raise ValueError(f'Unknown model: {model}')
    
    if not os.path.exists(path):
        print(f'  [MISSING] {path}')
        return None
    df = pd.read_csv(path)
    return df['accuracy'].values


def load_summary(model, split, k_shot, distance='euclidean'):
    """Load summary.json for a model/split/shot combination."""
    if model == 'ncc':
        path = os.path.join(
            RESULTS, 'baseline', 'resnet50', split, 'novel_ncc',
            f'{k_shot}shot_{distance}', 'summary.json'
        )
    elif model == 'protonet':
        results_split = 'base10_novel10' if '10' in split else 'base15_novel5'
        path = os.path.join(
            RESULTS, 'proto_net', results_split,
            f'{k_shot}shot_{distance}', 'summary.json'
        )
    else:
        raise ValueError(f'Unknown model: {model}')
    
    if not os.path.exists(path):
        print(f'  [MISSING] {path}')
        return None
    with open(path) as f:
        return json.load(f)

## 1. Aggregate accuracy comparison: B10/N10 vs B15/N5

In [ ]:
rows = []
for split_key, split_dir in [('B15/N5', 'base15'), ('B10/N10', 'base10')]:
    for model in ['ncc', 'protonet']:
        for k in [1, 3, 5]:
            s = load_summary(model, split_dir, k, 'euclidean')
            if s:
                rows.append({
                    'Split': split_key, 'Model': model.upper(),
                    'K-shot': k, 'Distance': 'Euclidean',
                    'Accuracy (%)': s['mean_accuracy_pct'],
                    'CI95': s['ci95_pct'],
                })

df_agg = pd.DataFrame(rows)
display(df_agg.pivot_table(
    index=['Model', 'Distance'],
    columns=['Split', 'K-shot'],
    values='Accuracy (%)',
).round(2))

## 2. Per-episode paired difference: NCC − ProtoNet at 5-shot B10/N10

In [ ]:
ncc_eps   = load_episodes('ncc',      'base10', 5, 'euclidean')
proto_eps = load_episodes('protonet', 'base10', 5, 'euclidean')

if ncc_eps is not None and proto_eps is not None:
    diff = ncc_eps - proto_eps
    mean_diff = diff.mean() * 100
    print(f'Mean NCC − ProtoNet per episode : {mean_diff:+.3f} pp')
    print(f'NCC wins   : {(diff > 0).sum()} / {len(diff)} episodes')
    print(f'ProtoNet wins: {(diff < 0).sum()} / {len(diff)} episodes')
    print(f'Ties       : {(diff == 0).sum()} / {len(diff)} episodes')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram of per-episode differences
    axes[0].hist(diff * 100, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[0].axvline(0, color='red', lw=1.5, ls='--', label='No difference')
    axes[0].axvline(mean_diff, color='navy', lw=1.5, ls='-', label=f'Mean = {mean_diff:+.2f} pp')
    axes[0].set_xlabel('NCC − ProtoNet accuracy (pp)')
    axes[0].set_ylabel('Episode count')
    axes[0].set_title('Per-episode accuracy difference\nNCC − ProtoNet (B10/N10, 5-shot Euclidean)')
    axes[0].legend()

    # Episode-level scatter plot
    axes[1].scatter(proto_eps * 100, ncc_eps * 100, alpha=0.15, s=10, color='steelblue')
    lims = [min(proto_eps.min(), ncc_eps.min()) * 100 - 2,
            max(proto_eps.max(), ncc_eps.max()) * 100 + 2]
    axes[1].plot(lims, lims, 'r--', lw=1, label='NCC = ProtoNet')
    axes[1].set_xlabel('ProtoNet accuracy (%)')
    axes[1].set_ylabel('NCC accuracy (%)')
    axes[1].set_title('Episode-level accuracy scatter\n(NCC vs ProtoNet, B10/N10 5-shot)')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(ROOT, 'figures', 'analysis_ncc_vs_proto_b10_5shot_diff.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

## 3. Welch's t-test: Is the NCC > ProtoNet reversal statistically significant?

In [ ]:
for k in [1, 3, 5]:
    ncc_e   = load_episodes('ncc',      'base10', k, 'euclidean')
    proto_e = load_episodes('protonet', 'base10', k, 'euclidean')
    if ncc_e is None or proto_e is None:
        continue
    t_stat, p_val = stats.ttest_ind(ncc_e, proto_e, equal_var=False)
    direction = 'NCC > ProtoNet' if ncc_e.mean() > proto_e.mean() else 'ProtoNet > NCC'
    sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'n.s.'))
    print(f'B10/N10 {k}-shot Eucl: NCC={ncc_e.mean()*100:.2f}% vs Proto={proto_e.mean()*100:.2f}%  '
          f't={t_stat:.3f}  p={p_val:.4f} {sig}  [{direction}]')

## 4. Per-class breakdown: Which novel classes drive the reversal?

In [ ]:
ncc_s5   = load_summary('ncc',      'base10', 5, 'euclidean')
proto_s5 = load_summary('protonet', 'base10', 5, 'euclidean')

if ncc_s5 and proto_s5:
    novel_classes = ncc_s5['novel_classes']
    ncc_pc   = ncc_s5['per_class_accuracy_pct']
    proto_pc = proto_s5['per_class_accuracy_pct']

    rows = []
    for cls in novel_classes:
        ncc_acc   = ncc_pc.get(cls, 0)
        proto_acc = proto_pc.get(cls, 0)
        rows.append({'Class': cls, 'NCC': ncc_acc, 'ProtoNet': proto_acc,
                     'NCC−ProtoNet': ncc_acc - proto_acc})

    df_pc = pd.DataFrame(rows).sort_values('NCC−ProtoNet', ascending=False)
    display(df_pc.round(2))

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(df_pc))
    bars = ax.bar(x, df_pc['NCC−ProtoNet'],
                  color=['#2196F3' if v >= 0 else '#F44336' for v in df_pc['NCC−ProtoNet']],
                  alpha=0.85)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([c.replace('AMERICAN ', 'AM.\n') for c in df_pc['Class']],
                       rotation=0, ha='center', fontsize=9)
    ax.set_ylabel('NCC − ProtoNet accuracy (pp)')
    ax.set_title('Per-class accuracy advantage of NCC over ProtoNet\n(B10/N10, 5-shot Euclidean)')
    plt.tight_layout()
    plt.savefig(os.path.join(ROOT, 'figures', 'analysis_ncc_vs_proto_per_class_b10.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

## 5. Three-factor explanation summary

The B10/N10 NCC > ProtoNet reversal arises from the interaction of three factors:

| Factor | Effect on ProtoNet | Effect on NCC |
|--------|-------------------|--------------|
| Reduced base training diversity (10 vs 15 classes) | Weaker episodic training → prototypes less discriminative | None: NCC uses ImageNet-pretrained features directly |
| NCC immunity to feature drift | N/A | NCC bypasses the episodic training phase entirely |
| Harder novel classes (10 vs 5) | More diverse confusions across 10-class pool | Centroid averaging over wider class distribution |

Key evidence:
- ProtoNet accuracy drops by **3.74 pp** (B15→B10 at 5-shot Euclidean: 99.80%→95.77%)
- NCC accuracy drops by only **0.74 pp** (98.14%→97.40%)
- The asymmetric degradation confirms that episodic training is **more sensitive** to base class diversity than centroid-based classification

## 6. Across all shot counts and distance metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, dist in zip(axes, ['euclidean', 'manhattan']):
    for model, color, label in [('ncc', '#1976D2', 'NCC'), ('protonet', '#E53935', 'ProtoNet')]:
        accs = []
        for k in [1, 3, 5]:
            s = load_summary(model, 'base10', k, dist)
            accs.append(s['mean_accuracy_pct'] if s else np.nan)
        ax.plot([1, 3, 5], accs, marker='o', color=color, label=label, lw=2)
    ax.set_xlabel('K-shot')
    ax.set_ylabel('Mean accuracy (%)')
    ax.set_title(f'B10/N10 — {dist.capitalize()} distance')
    ax.set_xticks([1, 3, 5])
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('NCC vs ProtoNet accuracy — B10/N10 split', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'figures', 'analysis_ncc_vs_proto_b10_kshot_trend.png'),
            dpi=150, bbox_inches='tight')
plt.show()